# Identity-preserving nonlinear evidence and custom SVC geometries

Clean research notebook. Tuning uses repeated CV on `train.csv`; external
`validation.csv` is evaluated only after CV selection. This notebook never fits
on validation, accesses test data, or writes a submission.


In [1]:
from pathlib import Path
from time import perf_counter
import gc
import random

import numpy as np
import pandas as pd
from IPython.display import display

from svc_new_experiments import (
    KernelSpec, build_combined_cache, build_repeated_cache,
    current_best_repeated, current_best_validation,
    disagreement_diagnostic, evidence_diagnostics,
    external_validation, external_validation_combined,
    optimize_cached, paired_stats, result_row,
)

SEED = 0xFACED
random.seed(SEED)
np.random.seed(SEED)


## Data loading and leakage guards


In [2]:
LOCAL = not Path('/kaggle/input').exists()
DATA_DIR = Path('./data') if LOCAL else Path('/kaggle/input/competitions/week-03-hidden-pairs')
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df = pd.read_csv(DATA_DIR / 'validation.csv')

feature_columns = [c for c in train_df.columns if c not in {'row_id', 'target'}]
assert 'row_id' not in feature_columns and 'target' not in feature_columns
assert feature_columns == [c for c in val_df.columns if c not in {'row_id', 'target'}]
X_train, y_train = train_df[feature_columns].copy(), train_df['target'].astype(int).copy()
X_val, y_val = val_df[feature_columns].copy(), val_df['target'].astype(int).copy()
for name, X in [('train', X_train), ('validation', X_val)]:
    assert X.shape[1] == 512 and np.isfinite(X.to_numpy()).all() and (X.to_numpy() >= 0).all(), name
print('train:', X_train.shape, 'validation:', X_val.shape)


train: (2520, 512) validation: (560, 512)


## Configuration


In [3]:
N_SPLITS = 5
CV_SEEDS = [SEED + i for i in range(5)]
N_BINS, ALPHA = 12, 5.0
N_JOBS = -1        # parallel scoring of the 25 cached folds
CACHE_N_JOBS = -1  # parallel construction of outer-fold caches

RUN_CURRENT_BEST = True
RUN_EVIDENCE = True
RUN_DIRECTION = True
RUN_CHI2 = True
RUN_LAPLACIAN = True
RUN_HELLINGER = True
RUN_HIST_INTERSECTION = False
RUN_EVIDENCE_PRODUCT = True
RUN_FINAL_COMBINATION = True

SINGLE_TRIALS, TWO_BLOCK_TRIALS = 40, 60
THREE_BLOCK_TRIALS, SECONDARY_TRIALS = 70, 30

# Only the fixed previous winner is reused; no old searches are transferred.
CURRENT_BEST_CONFIG = 'coord_monotonic'
CURRENT_BEST_PARAMS = {
    'kernel_type': 'product', 'C': 18.152351,
    'gamma_raw': 0.002496, 'gamma_geom': 0.010154,
}

records = {}

def run_experiment(cache, spec, trials, validation_family):
    started = perf_counter()
    _, params, aucs, oof = optimize_cached(cache, spec, trials, SEED, n_jobs=N_JOBS)
    val_auc, val_score = external_validation(
        X_train, y_train, X_val, y_val, validation_family,
        spec, params, SEED, N_BINS, ALPHA,
    )
    records[spec.name] = {
        'spec': spec, 'params': params, 'aucs': aucs, 'oof': oof,
        'validation_auc': val_auc, 'validation_score': val_score,
        'fit_time': perf_counter() - started,
    }
    print(spec.name, f'CV={aucs.mean():.6f}', f'validation={val_auc:.6f}')
    return records[spec.name]


## Baseline A — Raw log1p + StandardScaler + RBF SVC


In [4]:
raw_spec = KernelSpec('Baseline A — Raw RBF', 'Baseline', ('raw',), 'single')
raw_cache = build_repeated_cache(X_train, y_train, CV_SEEDS, 'raw', N_SPLITS, n_jobs=CACHE_N_JOBS)
raw_record = run_experiment(raw_cache, raw_spec, SINGLE_TRIALS, 'raw')
raw_aucs, raw_oof = raw_record['aucs'], raw_record['oof']
del raw_cache
gc.collect()


cache raw: repeat 3/5, fold 2/5cache raw: repeat 1/5, fold 5/5
cache raw: repeat 1/5, fold 3/5
cache raw: repeat 2/5, fold 2/5
cache raw: repeat 2/5, fold 4/5
cache raw: repeat 2/5, fold 1/5
cache raw: repeat 2/5, fold 5/5
cache raw: repeat 3/5, fold 1/5
cache raw: repeat 2/5, fold 3/5
cache raw: repeat 1/5, fold 4/5

cache raw: repeat 1/5, fold 2/5
cache raw: repeat 1/5, fold 1/5
cache raw: repeat 3/5, fold 3/5
cache raw: repeat 3/5, fold 4/5
cache raw: repeat 3/5, fold 5/5
cache raw: repeat 4/5, fold 1/5
cache raw: repeat 4/5, fold 2/5
cache raw: repeat 4/5, fold 3/5
cache raw: repeat 4/5, fold 4/5
cache raw: repeat 4/5, fold 5/5
cache raw: repeat 5/5, fold 1/5
cache raw: repeat 5/5, fold 2/5
cache raw: repeat 5/5, fold 3/5
cache raw: repeat 5/5, fold 4/5
cache raw: repeat 5/5, fold 5/5


c:\Users\peret\Desktop\Third year\ML\comp\comp 1\sbmt_kaggle\venvmy\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Best trial: 22. Best value: 0.929299: 100%|██████████| 40/40 [00:32<00:00,  1.24it/s]


Baseline A — Raw RBF CV=0.929299 validation=0.947423


1352

## Baseline B — previous Current Best on identical folds

This remains expensive because its supervised features are inner-OOF.


In [5]:
current_aucs = current_oof = None
current_validation_auc = np.nan
current_validation_score = None
if RUN_CURRENT_BEST:
    started = perf_counter()
    current_aucs, current_oof = current_best_repeated(
        X_train, y_train, CV_SEEDS, CURRENT_BEST_CONFIG, CURRENT_BEST_PARAMS, N_SPLITS,
    )
    current_validation_auc, current_validation_score = current_best_validation(
        X_train, y_train, X_val, y_val, CURRENT_BEST_CONFIG, CURRENT_BEST_PARAMS, SEED,
    )
    records['Baseline B — Current Best'] = {
        'spec': KernelSpec('Baseline B — Current Best', 'Baseline', ('current_best',), 'product'),
        'params': CURRENT_BEST_PARAMS, 'aucs': current_aucs, 'oof': current_oof,
        'validation_auc': current_validation_auc, 'validation_score': current_validation_score,
        'fit_time': perf_counter() - started,
    }
    print('Current Best:', current_aucs.mean(), current_validation_auc)


current best repeat=1, fold=1
current best repeat=1, fold=2
current best repeat=1, fold=3
current best repeat=1, fold=4
current best repeat=1, fold=5
current best repeat=2, fold=1
current best repeat=2, fold=2
current best repeat=2, fold=3
current best repeat=2, fold=4
current best repeat=2, fold=5
current best repeat=3, fold=1
current best repeat=3, fold=2
current best repeat=3, fold=3
current best repeat=3, fold=4
current best repeat=3, fold=5
current best repeat=4, fold=1
current best repeat=4, fold=2
current best repeat=4, fold=3
current best repeat=4, fold=4
current best repeat=4, fold=5
current best repeat=5, fold=1
current best repeat=5, fold=2
current best repeat=5, fold=3
current best repeat=5, fold=4
current best repeat=5, fold=5
Current Best: 0.931038674729151 0.9535714285714286


# Phase A — full 512-dimensional coordinate evidence


In [6]:
if RUN_EVIDENCE:
    evidence_feature_table, evidence_feature_summary, _ = evidence_diagnostics(
        X_train, y_train, X_val, y_val, SEED, N_BINS, ALPHA,
    )
    display(evidence_feature_summary)
    display(evidence_feature_table.sort_values('evidence_single_auc', ascending=False).head(30))


,representation,median_auc,max_auc,n_auc_gt_060,n_auc_gt_065,n_auc_gt_070
0,evidence_binned,0.628288,0.691794,471,66,0
1,evidence_isotonic,0.630991,0.694661,478,74,0


,representation,feature,raw_single_auc,evidence_single_auc,raw_spearman,raw_evidence_corr
683,evidence_isotonic,171,0.299765,0.694661,-0.346817,-0.986728
930,evidence_isotonic,418,0.301709,0.691952,-0.343450,-0.986082
171,evidence_binned,171,0.299765,0.691794,-0.346817,-0.975551
418,evidence_binned,418,0.301709,0.688857,-0.343450,-0.969209
801,evidence_isotonic,289,0.308495,0.683623,-0.331697,-0.985379
286,evidence_binned,286,0.309625,0.682455,-0.329740,-0.983205
634,evidence_isotonic,122,0.309310,0.682387,-0.330285,-0.988077
798,evidence_isotonic,286,0.309625,0.682025,-0.329740,-0.988182
879,evidence_isotonic,367,0.310167,0.681000,-0.328801,-0.981554
122,evidence_binned,122,0.309310,0.680299,-0.330285,-0.981584


In [7]:
if RUN_EVIDENCE:
    binned_cache = build_repeated_cache(
        X_train, y_train, CV_SEEDS, 'evidence_binned', N_SPLITS, N_BINS, ALPHA, n_jobs=CACHE_N_JOBS,
    )
    binned_specs = [
        (KernelSpec('A1 Evidence binned only', 'Evidence', ('evidence_binned',), 'single'), SINGLE_TRIALS),
        (KernelSpec('A3 Raw + binned concat', 'Evidence', ('concat_binned',), 'single'), SINGLE_TRIALS),
        (KernelSpec('A5 Raw + Binned Evidence additive', 'Evidence', ('raw', 'evidence_binned'), 'additive'), TWO_BLOCK_TRIALS),
    ]
    if RUN_EVIDENCE_PRODUCT:
        binned_specs.append((KernelSpec('A7 Raw + Binned Evidence product', 'Evidence', ('raw', 'evidence_binned'), 'product'), SECONDARY_TRIALS))
    for spec, trials in binned_specs:
        run_experiment(binned_cache, spec, trials, 'evidence_binned')
    del binned_cache
    gc.collect()


cache evidence_binned: repeat 2/5, fold 5/5
cache evidence_binned: repeat 2/5, fold 3/5
cache evidence_binned: repeat 2/5, fold 4/5
cache evidence_binned: repeat 3/5, fold 1/5
cache evidence_binned: repeat 3/5, fold 2/5
cache evidence_binned: repeat 1/5, fold 2/5
cache evidence_binned: repeat 2/5, fold 2/5
cache evidence_binned: repeat 1/5, fold 3/5
cache evidence_binned: repeat 1/5, fold 4/5
cache evidence_binned: repeat 2/5, fold 1/5
cache evidence_binned: repeat 1/5, fold 1/5
cache evidence_binned: repeat 1/5, fold 5/5
cache evidence_binned: repeat 3/5, fold 3/5
cache evidence_binned: repeat 3/5, fold 4/5
cache evidence_binned: repeat 3/5, fold 5/5
cache evidence_binned: repeat 4/5, fold 1/5
cache evidence_binned: repeat 4/5, fold 2/5
cache evidence_binned: repeat 4/5, fold 3/5
cache evidence_binned: repeat 4/5, fold 4/5
cache evidence_binned: repeat 4/5, fold 5/5
cache evidence_binned: repeat 5/5, fold 1/5
cache evidence_binned: repeat 5/5, fold 2/5
cache evidence_binned: repeat 5/

Best trial: 39. Best value: 0.918085: 100%|██████████| 40/40 [00:33<00:00,  1.20it/s]


A1 Evidence binned only CV=0.918085 validation=0.935510


Best trial: 16. Best value: 0.927218: 100%|██████████| 40/40 [00:33<00:00,  1.18it/s]


A3 Raw + binned concat CV=0.927218 validation=0.945383


Best trial: 45. Best value: 0.929278: 100%|██████████| 60/60 [01:04<00:00,  1.08s/it]


A5 Raw + Binned Evidence additive CV=0.929278 validation=0.947934


Best trial: 29. Best value: 0.92931: 100%|██████████| 30/30 [00:27<00:00,  1.08it/s] 


A7 Raw + Binned Evidence product CV=0.929310 validation=0.947883


In [8]:
if RUN_EVIDENCE:
    isotonic_cache = build_repeated_cache(
        X_train, y_train, CV_SEEDS, 'evidence_isotonic', N_SPLITS, N_BINS, ALPHA, n_jobs=CACHE_N_JOBS,
    )
    isotonic_specs = [
        (KernelSpec('A2 Evidence isotonic only', 'Evidence', ('evidence_isotonic',), 'single'), SINGLE_TRIALS),
        (KernelSpec('A4 Raw + isotonic concat', 'Evidence', ('concat_isotonic',), 'single'), SINGLE_TRIALS),
        (KernelSpec('A6 Raw + Isotonic Evidence additive', 'Evidence', ('raw', 'evidence_isotonic'), 'additive'), TWO_BLOCK_TRIALS),
    ]
    if RUN_EVIDENCE_PRODUCT:
        isotonic_specs.append((KernelSpec('A8 Raw + Isotonic Evidence product', 'Evidence', ('raw', 'evidence_isotonic'), 'product'), SECONDARY_TRIALS))
    for spec, trials in isotonic_specs:
        run_experiment(isotonic_cache, spec, trials, 'evidence_isotonic')
    del isotonic_cache
    gc.collect()


cache evidence_isotonic: repeat 2/5, fold 1/5
cache evidence_isotonic: repeat 1/5, fold 5/5
cache evidence_isotonic: repeat 2/5, fold 3/5
cache evidence_isotonic: repeat 2/5, fold 2/5
cache evidence_isotonic: repeat 2/5, fold 4/5
cache evidence_isotonic: repeat 1/5, fold 1/5
cache evidence_isotonic: repeat 3/5, fold 1/5
cache evidence_isotonic: repeat 3/5, fold 2/5
cache evidence_isotonic: repeat 1/5, fold 3/5
cache evidence_isotonic: repeat 1/5, fold 2/5
cache evidence_isotonic: repeat 2/5, fold 5/5
cache evidence_isotonic: repeat 1/5, fold 4/5
cache evidence_isotonic: repeat 3/5, fold 3/5
cache evidence_isotonic: repeat 3/5, fold 4/5
cache evidence_isotonic: repeat 3/5, fold 5/5
cache evidence_isotonic: repeat 4/5, fold 1/5
cache evidence_isotonic: repeat 4/5, fold 2/5
cache evidence_isotonic: repeat 4/5, fold 3/5
cache evidence_isotonic: repeat 4/5, fold 4/5
cache evidence_isotonic: repeat 4/5, fold 5/5
cache evidence_isotonic: repeat 5/5, fold 1/5
cache evidence_isotonic: repeat 5/

Best trial: 32. Best value: 0.899066: 100%|██████████| 40/40 [00:31<00:00,  1.29it/s]


A2 Evidence isotonic only CV=0.899066 validation=0.913406


Best trial: 23. Best value: 0.921441: 100%|██████████| 40/40 [00:31<00:00,  1.29it/s]


A4 Raw + isotonic concat CV=0.921441 validation=0.937781


Best trial: 48. Best value: 0.929071: 100%|██████████| 60/60 [01:04<00:00,  1.08s/it]


A6 Raw + Isotonic Evidence additive CV=0.929071 validation=0.947513


Best trial: 29. Best value: 0.929044: 100%|██████████| 30/30 [00:27<00:00,  1.09it/s]


A8 Raw + Isotonic Evidence product CV=0.929044 validation=0.947870


# Phase B — magnitude versus coordinate-preserving direction


In [9]:
if RUN_DIRECTION:
    direction_cache = build_repeated_cache(X_train, y_train, CV_SEEDS, 'direction', N_SPLITS, n_jobs=CACHE_N_JOBS)
    direction_specs = [
        (KernelSpec('B1 L2 Direction RBF', 'Direction', ('direction_l2',), 'single'), SINGLE_TRIALS),
        (KernelSpec('B2 L1 Composition RBF', 'Direction', ('composition_l1',), 'single'), SINGLE_TRIALS),
        (KernelSpec('B3 Raw + L2 Direction', 'Direction', ('raw', 'direction_l2'), 'additive'), TWO_BLOCK_TRIALS),
        (KernelSpec('B4 Raw + L1 Composition', 'Direction', ('raw', 'composition_l1'), 'additive'), TWO_BLOCK_TRIALS),
        (KernelSpec('B5 Raw + Direction + Magnitude', 'Direction', ('raw', 'direction_l2', 'magnitude'), 'additive'), THREE_BLOCK_TRIALS),
    ]
    for spec, trials in direction_specs:
        run_experiment(direction_cache, spec, trials, 'direction')
    del direction_cache
    gc.collect()


cache direction: repeat 1/5, fold 5/5cache direction: repeat 3/5, fold 2/5
cache direction: repeat 1/5, fold 3/5
cache direction: repeat 3/5, fold 1/5
cache direction: repeat 2/5, fold 5/5
cache direction: repeat 2/5, fold 2/5
cache direction: repeat 2/5, fold 4/5
cache direction: repeat 2/5, fold 1/5
cache direction: repeat 1/5, fold 2/5
cache direction: repeat 1/5, fold 4/5
cache direction: repeat 2/5, fold 3/5
cache direction: repeat 1/5, fold 1/5

cache direction: repeat 3/5, fold 3/5cache direction: repeat 3/5, fold 4/5

cache direction: repeat 3/5, fold 5/5
cache direction: repeat 4/5, fold 1/5
cache direction: repeat 4/5, fold 2/5
cache direction: repeat 4/5, fold 3/5
cache direction: repeat 4/5, fold 4/5
cache direction: repeat 4/5, fold 5/5
cache direction: repeat 5/5, fold 1/5
cache direction: repeat 5/5, fold 2/5
cache direction: repeat 5/5, fold 3/5
cache direction: repeat 5/5, fold 4/5
cache direction: repeat 5/5, fold 5/5


Best trial: 8. Best value: 0.894213: 100%|██████████| 40/40 [00:34<00:00,  1.16it/s]


B1 L2 Direction RBF CV=0.894213 validation=0.921952


Best trial: 39. Best value: 0.893563: 100%|██████████| 40/40 [00:32<00:00,  1.23it/s]


B2 L1 Composition RBF CV=0.893563 validation=0.917679


...

B3 Raw + L2 Direction CV=0.929712 validation=0.949082


Best trial: 40. Best value: 0.929971: 100%|██████████| 60/60 [01:07<00:00,  1.12s/it]


B4 Raw + L1 Composition CV=0.929971 validation=0.947997


Best trial: 50. Best value: 0.927803: 100%|██████████| 70/70 [02:00<00:00,  1.73s/it]


B5 Raw + Direction + Magnitude CV=0.927803 validation=0.943724


# Phase C — nonnegative custom-kernel geometry


In [10]:
robust_cache = build_repeated_cache(X_train, y_train, CV_SEEDS, 'custom_robust', N_SPLITS, n_jobs=CACHE_N_JOBS)
run_experiment(
    robust_cache,
    KernelSpec('C0 Robust-positive Euclidean RBF', 'Custom kernel', ('robust_euclidean',), 'single'),
    SINGLE_TRIALS, 'custom_robust',
)
del robust_cache
gc.collect()


cache custom_robust: repeat 1/5, fold 1/5cache custom_robust: repeat 1/5, fold 2/5
cache custom_robust: repeat 1/5, fold 5/5
cache custom_robust: repeat 2/5, fold 1/5
cache custom_robust: repeat 2/5, fold 2/5
cache custom_robust: repeat 2/5, fold 3/5
cache custom_robust: repeat 3/5, fold 2/5
cache custom_robust: repeat 3/5, fold 1/5
cache custom_robust: repeat 2/5, fold 4/5
cache custom_robust: repeat 1/5, fold 4/5
cache custom_robust: repeat 2/5, fold 5/5
cache custom_robust: repeat 1/5, fold 3/5

cache custom_robust: repeat 3/5, fold 3/5
cache custom_robust: repeat 3/5, fold 4/5
cache custom_robust: repeat 3/5, fold 5/5
cache custom_robust: repeat 4/5, fold 1/5
cache custom_robust: repeat 4/5, fold 2/5
cache custom_robust: repeat 4/5, fold 3/5
cache custom_robust: repeat 4/5, fold 4/5
cache custom_robust: repeat 4/5, fold 5/5
cache custom_robust: repeat 5/5, fold 1/5
cache custom_robust: repeat 5/5, fold 2/5
cache custom_robust: repeat 5/5, fold 3/5
cache custom_robust: repeat 5/5, f

Best trial: 27. Best value: 0.930225: 100%|██████████| 40/40 [00:31<00:00,  1.25it/s]


C0 Robust-positive Euclidean RBF CV=0.930225 validation=0.948214


1218

In [12]:
if RUN_CHI2:
    chi2_cache = build_repeated_cache(X_train, y_train, CV_SEEDS, 'custom_chi2', N_SPLITS, n_jobs=CACHE_N_JOBS)
    chi = run_experiment(
        chi2_cache, KernelSpec('C1 Chi-square SVC', 'Custom kernel', ('chi2',), 'single'),
        SINGLE_TRIALS, 'custom_chi2',
    )
    if chi['aucs'].mean() >= raw_aucs.mean() - 0.003:
        run_experiment(
            chi2_cache,
            KernelSpec('C5 Raw RBF + Chi-square', 'Custom kernel', ('raw', 'chi2'), 'additive'),
            TWO_BLOCK_TRIALS, 'custom_chi2',
        )
    else:
        print('C5 skipped: standalone chi-square is not close to Raw.')
    del chi2_cache
    gc.collect()


cache custom_chi2: repeat 2/5, fold 4/5cache custom_chi2: repeat 2/5, fold 3/5
cache custom_chi2: repeat 2/5, fold 5/5
cache custom_chi2: repeat 3/5, fold 1/5
cache custom_chi2: repeat 3/5, fold 2/5
cache custom_chi2: repeat 1/5, fold 1/5
cache custom_chi2: repeat 2/5, fold 2/5
cache custom_chi2: repeat 2/5, fold 1/5
cache custom_chi2: repeat 1/5, fold 4/5
cache custom_chi2: repeat 1/5, fold 5/5
cache custom_chi2: repeat 1/5, fold 2/5
cache custom_chi2: repeat 1/5, fold 3/5

cache custom_chi2: repeat 3/5, fold 3/5
cache custom_chi2: repeat 3/5, fold 4/5
cache custom_chi2: repeat 3/5, fold 5/5
cache custom_chi2: repeat 4/5, fold 1/5
cache custom_chi2: repeat 4/5, fold 2/5
cache custom_chi2: repeat 4/5, fold 3/5
cache custom_chi2: repeat 4/5, fold 4/5
cache custom_chi2: repeat 4/5, fold 5/5
cache custom_chi2: repeat 5/5, fold 1/5
cache custom_chi2: repeat 5/5, fold 2/5
cache custom_chi2: repeat 5/5, fold 3/5
cache custom_chi2: repeat 5/5, fold 4/5
cache custom_chi2: repeat 5/5, fold 5/5


Best trial: 6. Best value: 0.927368: 100%|██████████| 40/40 [00:44<00:00,  1.11s/it]


C1 Chi-square SVC CV=0.927368 validation=0.940893


Best trial: 41. Best value: 0.929448: 100%|██████████| 60/60 [01:32<00:00,  1.54s/it]


C5 Raw RBF + Chi-square CV=0.929448 validation=0.944605


In [13]:
if RUN_LAPLACIAN:
    lap_cache = build_repeated_cache(X_train, y_train, CV_SEEDS, 'custom_laplacian', N_SPLITS, n_jobs=CACHE_N_JOBS)
    run_experiment(
        lap_cache, KernelSpec('C2 Laplacian SVC', 'Custom kernel', ('laplacian',), 'single'),
        SECONDARY_TRIALS, 'custom_laplacian',
    )
    del lap_cache
    gc.collect()

if RUN_HELLINGER:
    hel_cache = build_repeated_cache(X_train, y_train, CV_SEEDS, 'custom_hellinger', N_SPLITS, n_jobs=CACHE_N_JOBS)
    run_experiment(
        hel_cache, KernelSpec('C3 Hellinger-composition SVC', 'Custom kernel', ('hellinger',), 'single'),
        SECONDARY_TRIALS, 'custom_hellinger',
    )
    del hel_cache
    gc.collect()

if RUN_HIST_INTERSECTION:
    hist_cache = build_repeated_cache(X_train, y_train, CV_SEEDS, 'custom_histogram', N_SPLITS, n_jobs=CACHE_N_JOBS)
    run_experiment(
        hist_cache,
        KernelSpec('C4 Histogram Intersection', 'Custom kernel', ('histogram',), 'single', ('histogram',)),
        SECONDARY_TRIALS, 'custom_histogram',
    )
    del hist_cache
    gc.collect()


cache custom_laplacian: repeat 1/5, fold 3/5cache custom_laplacian: repeat 1/5, fold 1/5
cache custom_laplacian: repeat 1/5, fold 4/5
cache custom_laplacian: repeat 2/5, fold 2/5
cache custom_laplacian: repeat 2/5, fold 3/5
cache custom_laplacian: repeat 2/5, fold 4/5
cache custom_laplacian: repeat 2/5, fold 5/5
cache custom_laplacian: repeat 3/5, fold 1/5
cache custom_laplacian: repeat 3/5, fold 2/5
cache custom_laplacian: repeat 1/5, fold 2/5

cache custom_laplacian: repeat 1/5, fold 5/5
cache custom_laplacian: repeat 2/5, fold 1/5
cache custom_laplacian: repeat 3/5, fold 3/5
cache custom_laplacian: repeat 3/5, fold 4/5
cache custom_laplacian: repeat 3/5, fold 5/5
cache custom_laplacian: repeat 4/5, fold 1/5
cache custom_laplacian: repeat 4/5, fold 2/5
cache custom_laplacian: repeat 4/5, fold 3/5
cache custom_laplacian: repeat 4/5, fold 4/5
cache custom_laplacian: repeat 4/5, fold 5/5
cache custom_laplacian: repeat 5/5, fold 1/5
cache custom_laplacian: repeat 5/5, fold 2/5
cache cust

Best trial: 28. Best value: 0.921465: 100%|██████████| 30/30 [00:34<00:00,  1.16s/it]


C2 Laplacian SVC CV=0.921465 validation=0.930497
cache custom_hellinger: repeat 1/5, fold 5/5
cache custom_hellinger: repeat 1/5, fold 4/5
cache custom_hellinger: repeat 2/5, fold 1/5
cache custom_hellinger: repeat 1/5, fold 3/5
cache custom_hellinger: repeat 2/5, fold 2/5
cache custom_hellinger: repeat 1/5, fold 1/5
cache custom_hellinger: repeat 2/5, fold 3/5
cache custom_hellinger: repeat 2/5, fold 4/5
cache custom_hellinger: repeat 2/5, fold 5/5
cache custom_hellinger: repeat 3/5, fold 1/5
cache custom_hellinger: repeat 1/5, fold 2/5
cache custom_hellinger: repeat 3/5, fold 2/5
cache custom_hellinger: repeat 3/5, fold 3/5
cache custom_hellinger: repeat 3/5, fold 4/5
cache custom_hellinger: repeat 3/5, fold 5/5
cache custom_hellinger: repeat 4/5, fold 1/5
cache custom_hellinger: repeat 4/5, fold 2/5
cache custom_hellinger: repeat 4/5, fold 3/5
cache custom_hellinger: repeat 4/5, fold 4/5
cache custom_hellinger: repeat 4/5, fold 5/5
cache custom_hellinger: repeat 5/5, fold 1/5
cache 

Best trial: 23. Best value: 0.838502: 100%|██████████| 30/30 [00:32<00:00,  1.08s/it]


C3 Hellinger-composition SVC CV=0.838502 validation=0.857028


# Phase D — gated combination of successful ideas only


In [14]:
component_map = {
    'A5 Raw + Binned Evidence additive': 'evidence_binned',
    'A6 Raw + Isotonic Evidence additive': 'evidence_isotonic',
    'B3 Raw + L2 Direction': 'direction_l2',
    'B4 Raw + L1 Composition': 'composition_l1',
    'C5 Raw RBF + Chi-square': 'chi2',
}

successful_components = []
if current_aucs is not None:
    for experiment, block in component_map.items():
        if experiment not in records:
            continue
        record = records[experiment]
        paired = paired_stats(record['aucs'], current_aucs)
        if (paired['mean_delta'] >= 0.0015 and paired['wins'] >= 17 and
                record['validation_auc'] >= current_validation_auc - 0.001):
            successful_components.append((record['aucs'].mean(), experiment, block))
successful_components.sort(reverse=True)
print('Phase-D eligible components:', successful_components)

if RUN_FINAL_COMBINATION and len({x[2] for x in successful_components}) >= 2:
    chosen = []
    for _, experiment, block in successful_components:
        if block not in chosen:
            chosen.append(block)
        if len(chosen) == 2:
            break
    spaces = ('raw', *chosen)
    direct = tuple(s for s in spaces if s == 'histogram')
    final_spec = KernelSpec('D1 Gated three-block combination', 'Gated combination', spaces, 'additive', direct)
    combined_cache = build_combined_cache(X_train, y_train, CV_SEEDS, spaces, N_SPLITS, N_BINS, ALPHA, n_jobs=CACHE_N_JOBS)
    started = perf_counter()
    _, params, aucs, oof = optimize_cached(combined_cache, final_spec, THREE_BLOCK_TRIALS, SEED, n_jobs=N_JOBS)
    val_auc, val_score = external_validation_combined(
        X_train, y_train, X_val, y_val, spaces, final_spec, params, SEED, N_BINS, ALPHA,
    )
    records[final_spec.name] = {
        'spec': final_spec, 'params': params, 'aucs': aucs, 'oof': oof,
        'validation_auc': val_auc, 'validation_score': val_score,
        'fit_time': perf_counter() - started,
    }
    del combined_cache
    gc.collect()
else:
    print('Phase D skipped: fewer than two distinct blocks passed the stability gate.')


Phase-D eligible components: []
Phase D skipped: fewer than two distinct blocks passed the stability gate.


# Paired results, family summaries and diagnostics


In [15]:
rows = []
for name, record in records.items():
    spec = record['spec']
    rows.append(result_row(
        name, spec.family, ' + '.join(spec.spaces), spec.combination,
        record['aucs'], record['validation_auc'], record['oof'],
        raw_aucs, raw_oof, current_aucs, current_oof,
        record['fit_time'], record['params'],
    ))
results = pd.DataFrame(rows).sort_values('CV_mean', ascending=False).reset_index(drop=True)
main_columns = [
    'experiment', 'family', 'representation', 'kernel',
    'CV_mean', 'CV_std', 'CV_median', 'delta_vs_raw', 'delta_vs_current',
    'wins_vs_raw', 'losses_vs_raw', 'wins_vs_current', 'losses_vs_current',
    'wilcoxon_vs_raw', 'wilcoxon_vs_current', 'validation_auc',
    'corr_with_raw', 'corr_with_current', 'fit_time', 'params',
]
display(results[main_columns])

foldwise_auc = pd.DataFrame({name: record['aucs'] for name, record in records.items()})
foldwise_auc.insert(0, 'fold_id', [
    f'seed_{r}_fold_{f}' for r in range(len(CV_SEEDS)) for f in range(N_SPLITS)
])
display(foldwise_auc)


,experiment,family,representation,kernel,CV_mean,CV_std,CV_median,delta_vs_raw,delta_vs_current,wins_vs_raw,losses_vs_raw,wins_vs_current,losses_vs_current,wilcoxon_vs_raw,wilcoxon_vs_current,validation_auc,corr_with_raw,corr_with_current,fit_time,params
0,Baseline B — Current Best,Baseline,current_best,product,0.931039,0.007146,0.929910,0.001740,0.000000,18,7,0,0,1.625359e-03,1.000000e+00,0.953571,0.986188,1.000000,604.630484,"{'kernel_type': 'product', 'C': 18.152351, 'ga..."
1,C0 Robust-positive Euclidean RBF,Custom kernel,robust_euclidean,single,0.930225,0.007783,0.929359,0.000926,-0.000814,17,8,10,15,2.255261e-03,1.729586e-01,0.948214,0.996133,0.981914,36.937866,"{'C': 29.325916578196896, 'gamma_robust_euclid..."
2,B4 Raw + L1 Composition,Direction,raw + composition_l1,additive,0.929971,0.007549,0.928997,0.000672,-0.001068,16,9,8,17,1.923179e-02,3.933871e-02,0.947997,0.997522,0.984358,81.775493,"{'C': 19.342934816919954, 'gamma_raw': 0.00275..."
3,B3 Raw + L2 Direction,Direction,raw + direction_l2,additive,0.929712,0.007513,0.928965,0.000413,-0.001327,16,9,8,17,1.425140e-01,1.874471e-02,0.949082,0.996734,0.982981,78.707176,"{'C': 11.37773717425642, 'gamma_raw': 0.003049..."
4,C5 Raw RBF + Chi-square,Custom kernel,raw + chi2,additive,0.929448,0.008028,0.929674,0.000149,-0.001591,11,14,6,19,9.368029e-01,8.822322e-03,0.944605,0.991748,0.977800,175.348357,"{'C': 13.472034418791413, 'gamma_raw': 0.00225..."
5,A7 Raw + Binned Evidence product,Evidence,raw + evidence_binned,product,0.929310,0.007402,0.928193,0.000011,-0.001728,11,14,5,20,7.878671e-01,3.290772e-04,0.947883,0.999445,0.986875,49.183444,"{'C': 10.558627633121196, 'gamma_raw': 0.00271..."
6,Baseline A — Raw RBF,Baseline,raw,single,0.929299,0.007345,0.928886,0.000000,-0.001740,0,0,7,18,1.000000e+00,1.625359e-03,0.947423,1.000000,0.986188,37.251800,"{'C': 2.3212580455740537, 'gamma_raw': 0.00282..."
7,A5 Raw + Binned Evidence additive,Evidence,raw + evidence_binned,additive,0.929278,0.007398,0.928493,-0.000021,-0.001761,10,15,4,21,6.720752e-01,2.498031e-04,0.947934,0.998832,0.986609,85.834125,"{'C': 29.03611375476341, 'gamma_raw': 0.002758..."
8,A6 Raw + Isotonic Evidence additive,Evidence,raw + evidence_isotonic,additive,0.929071,0.007478,0.927532,-0.000228,-0.001968,8,17,4,21,1.784940e-01,4.541874e-05,0.947513,0.999211,0.987075,90.916558,"{'C': 4.754391942406471, 'gamma_raw': 0.002516..."
9,A8 Raw + Isotonic Evidence product,Evidence,raw + evidence_isotonic,product,0.929044,0.007556,0.928225,-0.000255,-0.001995,7,18,3,22,1.873369e-01,2.157118e-04,0.947870,0.998680,0.986933,53.867948,"{'C': 17.326822484481465, 'gamma_raw': 0.00249..."


,fold_id,Baseline A — Raw RBF,Baseline B — Current Best,A1 Evidence binned only,A3 Raw + binned concat,A5 Raw + Binned Evidence additive,A7 Raw + Binned Evidence product,A2 Evidence isotonic only,A4 Raw + isotonic concat,A6 Raw + Isotonic Evidence additive,...,B1 L2 Direction RBF,B2 L1 Composition RBF,B3 Raw + L2 Direction,B4 Raw + L1 Composition,B5 Raw + Direction + Magnitude,C0 Robust-positive Euclidean RBF,C1 Chi-square SVC,C5 Raw RBF + Chi-square,C2 Laplacian SVC,C3 Hellinger-composition SVC
0,seed_0_fold_0,0.923517,0.925076,0.914934,0.921973,0.924871,0.923406,0.891818,0.917218,0.924477,...,0.888479,0.890479,0.924241,0.924965,0.923532,0.924367,0.923139,0.927375,0.917501,0.823917
1,seed_0_fold_1,0.934697,0.938665,0.925391,0.934933,0.934603,0.935232,0.904164,0.929091,0.934587,...,0.899613,0.900967,0.935973,0.936728,0.934540,0.936287,0.932760,0.933957,0.925973,0.839616
2,seed_0_fold_2,0.929123,0.932335,0.920824,0.927201,0.928997,0.928729,0.904274,0.923328,0.928414,...,0.904982,0.904006,0.929343,0.929375,0.926839,0.930052,0.928713,0.929941,0.921186,0.844010
3,seed_0_fold_3,0.931453,0.936177,0.915155,0.926383,0.931233,0.931469,0.908336,0.925154,0.931217,...,0.891062,0.889566,0.929894,0.931138,0.927343,0.931311,0.929658,0.931059,0.922666,0.831570
4,seed_0_fold_4,0.925768,0.926320,0.914068,0.924351,0.924855,0.925737,0.888794,0.916383,0.925170,...,0.887550,0.886684,0.925532,0.925359,0.921281,0.925564,0.920777,0.924068,0.919501,0.834451
5,seed_1_fold_0,0.944885,0.945074,0.926666,0.939516,0.943405,0.944633,0.905408,0.933721,0.944114,...,0.922682,0.920367,0.943515,0.944649,0.945248,0.947090,0.944130,0.944996,0.938177,0.859269
6,seed_1_fold_1,0.935689,0.935784,0.929847,0.935185,0.935752,0.935736,0.918131,0.933815,0.935374,...,0.909329,0.912683,0.937028,0.937783,0.936996,0.936681,0.938539,0.938177,0.932870,0.840514
7,seed_1_fold_2,0.929327,0.932776,0.916919,0.926698,0.931264,0.930067,0.902841,0.922698,0.931201,...,0.895707,0.894841,0.931532,0.930776,0.926430,0.931359,0.926587,0.932319,0.917643,0.862024
8,seed_1_fold_3,0.926446,0.925202,0.908982,0.921485,0.925721,0.926288,0.881582,0.914273,0.925942,...,0.887755,0.887109,0.925406,0.926146,0.923391,0.927044,0.923973,0.925894,0.919564,0.818657
9,seed_1_fold_4,0.913848,0.916399,0.902904,0.910950,0.913139,0.913564,0.883692,0.904636,0.913045,...,0.856607,0.856781,0.912604,0.912557,0.906510,0.913517,0.908840,0.911880,0.901817,0.818106


In [16]:
for family in ['Evidence', 'Direction', 'Custom kernel', 'Gated combination']:
    family_table = results[results.family.isin(['Baseline', family])]
    if len(family_table):
        print(f'--- {family} ---')
        display(family_table[main_columns].reset_index(drop=True))


--- Evidence ---


,experiment,family,representation,kernel,CV_mean,CV_std,CV_median,delta_vs_raw,delta_vs_current,wins_vs_raw,losses_vs_raw,wins_vs_current,losses_vs_current,wilcoxon_vs_raw,wilcoxon_vs_current,validation_auc,corr_with_raw,corr_with_current,fit_time,params
0,Baseline B — Current Best,Baseline,current_best,product,0.931039,0.007146,0.929910,0.001740,0.000000,18,7,0,0,1.625359e-03,1.000000e+00,0.953571,0.986188,1.000000,604.630484,"{'kernel_type': 'product', 'C': 18.152351, 'ga..."
1,A7 Raw + Binned Evidence product,Evidence,raw + evidence_binned,product,0.929310,0.007402,0.928193,0.000011,-0.001728,11,14,5,20,7.878671e-01,3.290772e-04,0.947883,0.999445,0.986875,49.183444,"{'C': 10.558627633121196, 'gamma_raw': 0.00271..."
2,Baseline A — Raw RBF,Baseline,raw,single,0.929299,0.007345,0.928886,0.000000,-0.001740,0,0,7,18,1.000000e+00,1.625359e-03,0.947423,1.000000,0.986188,37.251800,"{'C': 2.3212580455740537, 'gamma_raw': 0.00282..."
3,A5 Raw + Binned Evidence additive,Evidence,raw + evidence_binned,additive,0.929278,0.007398,0.928493,-0.000021,-0.001761,10,15,4,21,6.720752e-01,2.498031e-04,0.947934,0.998832,0.986609,85.834125,"{'C': 29.03611375476341, 'gamma_raw': 0.002758..."
4,A6 Raw + Isotonic Evidence additive,Evidence,raw + evidence_isotonic,additive,0.929071,0.007478,0.927532,-0.000228,-0.001968,8,17,4,21,1.784940e-01,4.541874e-05,0.947513,0.999211,0.987075,90.916558,"{'C': 4.754391942406471, 'gamma_raw': 0.002516..."
5,A8 Raw + Isotonic Evidence product,Evidence,raw + evidence_isotonic,product,0.929044,0.007556,0.928225,-0.000255,-0.001995,7,18,3,22,1.873369e-01,2.157118e-04,0.947870,0.998680,0.986933,53.867948,"{'C': 17.326822484481465, 'gamma_raw': 0.00249..."
6,A3 Raw + binned concat,Evidence,concat_binned,single,0.927218,0.007324,0.926698,-0.002081,-0.003821,4,20,1,24,1.403786e-04,2.980232e-07,0.945383,0.992960,0.977603,60.431142,"{'C': 5.5947992128994795, 'gamma_concat_binned..."
7,A4 Raw + isotonic concat,Evidence,concat_isotonic,single,0.921441,0.008088,0.921060,-0.007858,-0.009598,0,25,0,25,5.960464e-08,5.960464e-08,0.937781,0.980608,0.967393,58.228556,"{'C': 2.1582417561626532, 'gamma_concat_isoton..."
8,A1 Evidence binned only,Evidence,evidence_binned,single,0.918085,0.007426,0.917926,-0.011214,-0.012954,0,25,0,25,5.960464e-08,5.960464e-08,0.935510,0.970741,0.953231,62.888966,"{'C': 16.128847697217076, 'gamma_evidence_binn..."
9,A2 Evidence isotonic only,Evidence,evidence_isotonic,single,0.899066,0.010596,0.901628,-0.030233,-0.031973,0,25,0,25,5.960464e-08,5.960464e-08,0.913406,0.931000,0.913418,58.859053,"{'C': 1.7991650448878298, 'gamma_evidence_isot..."


--- Direction ---


,experiment,family,representation,kernel,CV_mean,CV_std,CV_median,delta_vs_raw,delta_vs_current,wins_vs_raw,losses_vs_raw,wins_vs_current,losses_vs_current,wilcoxon_vs_raw,wilcoxon_vs_current,validation_auc,corr_with_raw,corr_with_current,fit_time,params
0,Baseline B — Current Best,Baseline,current_best,product,0.931039,0.007146,0.929910,0.001740,0.000000,18,7,0,0,1.625359e-03,1.000000e+00,0.953571,0.986188,1.000000,604.630484,"{'kernel_type': 'product', 'C': 18.152351, 'ga..."
1,B4 Raw + L1 Composition,Direction,raw + composition_l1,additive,0.929971,0.007549,0.928997,0.000672,-0.001068,16,9,8,17,1.923179e-02,3.933871e-02,0.947997,0.997522,0.984358,81.775493,"{'C': 19.342934816919954, 'gamma_raw': 0.00275..."
2,B3 Raw + L2 Direction,Direction,raw + direction_l2,additive,0.929712,0.007513,0.928965,0.000413,-0.001327,16,9,8,17,1.425140e-01,1.874471e-02,0.949082,0.996734,0.982981,78.707176,"{'C': 11.37773717425642, 'gamma_raw': 0.003049..."
3,Baseline A — Raw RBF,Baseline,raw,single,0.929299,0.007345,0.928886,0.000000,-0.001740,0,0,7,18,1.000000e+00,1.625359e-03,0.947423,1.000000,0.986188,37.251800,"{'C': 2.3212580455740537, 'gamma_raw': 0.00282..."
4,B5 Raw + Direction + Magnitude,Direction,raw + direction_l2 + magnitude,additive,0.927803,0.008182,0.927327,-0.001496,-0.003236,8,17,7,18,8.699023e-03,3.290772e-04,0.943724,0.978565,0.963393,134.052278,"{'C': 5.433573787061393, 'gamma_raw': 0.000110..."
5,B1 L2 Direction RBF,Direction,direction_l2,single,0.894213,0.011942,0.894148,-0.035086,-0.036825,0,25,0,25,5.960464e-08,5.960464e-08,0.921952,0.872204,0.873184,47.425462,"{'C': 29.22067102492162, 'gamma_direction_l2':..."
6,B2 L1 Composition RBF,Direction,composition_l1,single,0.893563,0.012086,0.893424,-0.035736,-0.037476,0,25,0,25,5.960464e-08,5.960464e-08,0.917679,0.873203,0.873037,45.266310,"{'C': 16.128847697217076, 'gamma_composition_l..."


--- Custom kernel ---


,experiment,family,representation,kernel,CV_mean,CV_std,CV_median,delta_vs_raw,delta_vs_current,wins_vs_raw,losses_vs_raw,wins_vs_current,losses_vs_current,wilcoxon_vs_raw,wilcoxon_vs_current,validation_auc,corr_with_raw,corr_with_current,fit_time,params
0,Baseline B — Current Best,Baseline,current_best,product,0.931039,0.007146,0.929910,0.001740,0.000000,18,7,0,0,1.625359e-03,1.000000e+00,0.953571,0.986188,1.000000,604.630484,"{'kernel_type': 'product', 'C': 18.152351, 'ga..."
1,C0 Robust-positive Euclidean RBF,Custom kernel,robust_euclidean,single,0.930225,0.007783,0.929359,0.000926,-0.000814,17,8,10,15,2.255261e-03,1.729586e-01,0.948214,0.996133,0.981914,36.937866,"{'C': 29.325916578196896, 'gamma_robust_euclid..."
2,C5 Raw RBF + Chi-square,Custom kernel,raw + chi2,additive,0.929448,0.008028,0.929674,0.000149,-0.001591,11,14,6,19,9.368029e-01,8.822322e-03,0.944605,0.991748,0.977800,175.348357,"{'C': 13.472034418791413, 'gamma_raw': 0.00225..."
3,Baseline A — Raw RBF,Baseline,raw,single,0.929299,0.007345,0.928886,0.000000,-0.001740,0,0,7,18,1.000000e+00,1.625359e-03,0.947423,1.000000,0.986188,37.251800,"{'C': 2.3212580455740537, 'gamma_raw': 0.00282..."
4,C1 Chi-square SVC,Custom kernel,chi2,single,0.927368,0.008356,0.927170,-0.001931,-0.003671,3,22,2,23,2.507925e-03,1.507998e-05,0.940893,0.983890,0.967450,125.982463,"{'C': 3.4476061667306257, 'gamma_chi2': 0.0141..."
5,C2 Laplacian SVC,Custom kernel,laplacian,single,0.921465,0.008738,0.921186,-0.007834,-0.009574,0,25,0,25,5.960464e-08,5.960464e-08,0.930497,0.978489,0.957871,48.483423,"{'C': 2.884825755142345, 'gamma_laplacian': 0...."
6,C3 Hellinger-composition SVC,Custom kernel,hellinger,single,0.838502,0.014872,0.837522,-0.090797,-0.092537,0,25,0,25,5.960464e-08,5.960464e-08,0.857028,0.779554,0.772692,43.426386,"{'C': 22.60321162551724, 'gamma_hellinger': 0...."


--- Gated combination ---


,experiment,family,representation,kernel,CV_mean,CV_std,CV_median,delta_vs_raw,delta_vs_current,wins_vs_raw,losses_vs_raw,wins_vs_current,losses_vs_current,wilcoxon_vs_raw,wilcoxon_vs_current,validation_auc,corr_with_raw,corr_with_current,fit_time,params
0,Baseline B — Current Best,Baseline,current_best,product,0.931039,0.007146,0.929910,0.00174,0.00000,18,7,0,0,0.001625,1.000000,0.953571,0.986188,1.000000,604.630484,"{'kernel_type': 'product', 'C': 18.152351, 'ga..."
1,Baseline A — Raw RBF,Baseline,raw,single,0.929299,0.007345,0.928886,0.00000,-0.00174,0,0,7,18,1.000000,0.001625,0.947423,1.000000,0.986188,37.251800,"{'C': 2.3212580455740537, 'gamma_raw': 0.00282..."


In [17]:
if current_oof is not None:
    new_models = results[~results.family.eq('Baseline')]
    best_new_name = new_models.iloc[0].experiment
    disagreement = disagreement_diagnostic(records[best_new_name]['oof'], current_oof, y_train)
    print('Disagreement diagnostic:', best_new_name)
    display(pd.DataFrame([disagreement]))


Disagreement diagnostic: C0 Robust-positive Euclidean RBF


,subset_size,new_auc,current_auc,mean_rank_diff
0,3156,0.941335,0.939133,0.091344


# Final research conclusions — no submission


In [18]:
best_cv = results.iloc[0]
best_validation = results.sort_values('validation_auc', ascending=False).iloc[0]
if current_aucs is not None:
    stable = results.sort_values(['delta_vs_current', 'wins_vs_current'], ascending=False).iloc[0]
    breakthrough = results[
        (results.delta_vs_current >= 0.002) &
        (results.wins_vs_current >= 18) &
        (results.validation_auc >= current_validation_auc - 0.001)
    ]
else:
    stable, breakthrough = None, results.iloc[0:0]

strong = results[(results.CV_mean >= raw_aucs.mean() - 0.002) & ~results.family.eq('Baseline')]
complementary = strong.sort_values('corr_with_current').iloc[0] if len(strong) and current_oof is not None else None

print('BEST REPEATED-CV MODEL:', best_cv.experiment, f'{best_cv.CV_mean:.6f}')
print('BEST VALIDATION MODEL:', best_validation.experiment, f'{best_validation.validation_auc:.6f}')
if stable is not None:
    print('BEST STABLE DELTA VS CURRENT:', stable.experiment,
          f'{stable.delta_vs_current:+.6f}', f'wins={int(stable.wins_vs_current)}/25')
if complementary is not None:
    print('MOST COMPLEMENTARY STRONG MODEL:', complementary.experiment,
          f'corr={complementary.corr_with_current:.5f}')
if len(breakthrough):
    print('BREAKTHROUGH CANDIDATES')
    display(breakthrough[main_columns])
else:
    print('NO BREAKTHROUGH CANDIDATE')
print('No validation training, test prediction, or submission was performed.')


BEST REPEATED-CV MODEL: Baseline B — Current Best 0.931039
BEST VALIDATION MODEL: Baseline B — Current Best 0.953571
BEST STABLE DELTA VS CURRENT: Baseline B — Current Best +0.000000 wins=0/25
MOST COMPLEMENTARY STRONG MODEL: B5 Raw + Direction + Magnitude corr=0.96339
NO BREAKTHROUGH CANDIDATE
No validation training, test prediction, or submission was performed.
